# DS-AL: Colab / Kaggle

Bật GPU trong runtime settings. Notebook gọi cùng mã nguồn của repository, không chép lại core method. Upload `artifacts/project-source.zip` do `python -m scripts.package_source` tạo. Kết quả offline cần sync W&B công khai trước khi nộp. Không nhúng token vào notebook.


In [ ]:
from pathlib import Path
import sys, zipfile, os, subprocess
try:
    from google.colab import files
    uploaded = files.upload()
    archive = next(Path(name) for name in uploaded if name.endswith('.zip'))
except ImportError:
    archives = list(Path('/kaggle/input').rglob('project-source.zip'))
    if not archives: raise RuntimeError('Attach project-source.zip as a Kaggle input dataset')
    archive = archives[0]
workspace = Path('/content/dsal-group3' if Path('/content').exists() else '/kaggle/working/dsal-group3')
workspace.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        target = (workspace / member.filename).resolve()
        if workspace.resolve() not in target.parents: raise ValueError('Unsafe archive member')
    z.extractall(workspace)
os.chdir(workspace)
print(workspace)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('If pip changes a package already imported, restart the runtime before continuing.')


## Tracking

Điền entity không bí mật. `wandb.login()` nhập API key qua ô bảo mật hoặc dùng secrets của nền tảng. Không ghi token vào cell. Đổi TRACKING thành `online` sau khi đăng nhập; dùng cùng project với máy local.


In [ ]:
ENTITY = ''
PROJECT = 'intro-ml-group3-dsal'
TRACKING = 'offline'
if ENTITY: os.environ['WANDB_ENTITY'] = ENTITY
os.environ['WANDB_PROJECT'] = PROJECT
if TRACKING == 'online':
    import wandb
    wandb.login()


In [ ]:
subprocess.run([sys.executable, '-m', 'scripts.doctor'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.run', '--config', 'configs/cifar100_smoke.json', '--stage', 'suite', '--tracking', TRACKING], check=True)


## Tái hiện đầy đủ trên CIFAR-100

Cấu hình mặc định160 epochs, buffer8192, K5/25/50. Runtime có thể dài. Checkpoint mỗi epoch. Để khôi phục sau khi runtime reset, tải lại checkpoint và feature cache vào cùng đường dẫn. Giữ nguyên config để tránh dùng nhầm checkpoint. Bộ nhớ buffer8192 gồm nhiều ma trận lớn.


In [ ]:
subprocess.run([sys.executable, '-m', 'scripts.run', '--config', 'configs/cifar100_core.json', '--stage', 'suite', '--tracking', TRACKING], check=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'scripts.empirics', '--tracking', TRACKING], check=True)
subprocess.run([sys.executable, '-m', 'scripts.summarize'], check=True)


## Ba seed và lưu kết quả

Chỉ bật RUN_THREE_SEEDS khi runtime/quota đủ. Ba seed có backbone riêng. Archive kết quả có log W&B offline, checkpoint và feature cache; dung lượng có thể lớn. Không chứa API key hoặc `.env`.


In [ ]:
RUN_THREE_SEEDS = False
if RUN_THREE_SEEDS:
    subprocess.run([sys.executable, '-m', 'scripts.multiseed', '--seeds', '0', '1', '2', '--tracking', TRACKING], check=True)
import shutil
shutil.make_archive(str(workspace.parent / 'dsal-results'), 'zip', workspace, 'outputs')
try:
    from google.colab import files
    files.download(str(workspace.parent / 'dsal-results.zip'))
except ImportError:
    print('Kaggle output:', workspace.parent / 'dsal-results.zip')
